# MNIST and Fashion-MNIST vanilla GAN

Both runs call the same engineer, split, train, and eval functions. The only change is which images `scripts/mnist_gan_comparison/pull_sample_data.py` wrote into `data/mnist_gan_comparison/` before that run. Archives live in `reports/mnist_gan_comparison/generations/digits/` and `reports/mnist_gan_comparison/generations/fashion/`.

The loss curves are discriminator and generator binary cross-entropy. The epoch grids are one fixed noise batch. A grid that turns into a single repeated image is mode collapse. `sample_std` in the table is the mean per-pixel deviation across that batch, so a repeated high-contrast shape scores low. The curve under the losses is the pixel-value std recorded during training, which can stay large after the shapes have collapsed.

Run this notebook from the repository root after `scripts/mnist_gan_comparison/train_comparison.py`.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display


def project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    return here


ROOT = project_root()
DATASETS = ("digits", "fashion")


def load_json(dataset: str, name: str) -> dict:
    path = ROOT / "reports" / "mnist_gan_comparison" / "generations" / dataset / name
    return json.loads(path.read_text(encoding="utf-8"))


histories = {name: load_json(name, "history.json") for name in DATASETS}
tests = {name: load_json(name, "evaluation_test.json") for name in DATASETS}
sources = {name: load_json(name, "source.json") for name in DATASETS}


: 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for axis, name in zip(axes, DATASETS, strict=True):
    history = histories[name]
    axis.plot(history["train_discriminator"], label="train D")
    axis.plot(history["train_generator"], label="train G")
    axis.plot(history["eval_discriminator"], label="eval D", linestyle="--")
    axis.plot(history["eval_generator"], label="eval G", linestyle="--")
    axis.set_title(name)
    axis.set_xlabel("epoch")
    axis.set_ylabel("bce")
    axis.legend(fontsize=8)
fig.suptitle("Generator and discriminator loss")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for axis, name in zip(axes, DATASETS, strict=True):
    axis.plot(histories[name]["sample_std"], marker="o")
    axis.axhline(tests[name]["real_std"], color="gray", linestyle=":", label="real std")
    axis.set_title(f"{name} logged pixel std")
    axis.set_xlabel("epoch")
    axis.set_ylabel("pixel std")
    axis.legend(fontsize=8)
fig.tight_layout()
plt.show()


In [ ]:
def show_epochs(dataset: str) -> None:
    folder = ROOT / "reports" / "mnist_gan_comparison" / "generations" / dataset
    frames = sorted(folder.glob("epoch_*.png"))
    if not frames:
        display(Markdown(f"**{dataset}:** no epoch grids"))
        return
    picks = [frames[0], frames[len(frames) // 2], frames[-1]]
    display(Markdown(f"### {dataset}"))
    for path in picks:
        display(Markdown(path.name))
        display(Image(filename=str(path)))
    display(Markdown("final fixed-noise batch and real test images"))
    display(Image(filename=str(folder / "final.png")))
    display(Image(filename=str(folder / "real_test.png")))


for dataset in DATASETS:
    show_epochs(dataset)


In [ ]:
rows = []
for name in DATASETS:
    metrics = tests[name]
    source = sources[name]
    rows.append(
        {
            "dataset": name,
            "epochs": source["epochs"],
            "train_per_class": source["train_per_class"],
            "discriminator_loss": round(metrics["discriminator_loss"], 4),
            "generator_loss": round(metrics["generator_loss"], 4),
            "sample_std": round(metrics["sample_std"], 4),
            "real_std": round(metrics["real_std"], 4),
            "mode_collapse": metrics["mode_collapse"],
        }
    )
rows
